# 06 Deep learning: GRU and LSTM
**Design fixed before seeing results:** input = last 168 h (load ÷ window mean + calendar), plus the target day's 24 calendar rows; output = 24 ratios; fixed 60 epochs, L1 loss, 3 seeds averaged, no tuning on test folds.

**Needs:** PyTorch (CPU build) in the environment. It is used only in this notebook, not by the app.

In [1]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, torch, torch.nn as nn
import forecast_lib as fl

h = pd.read_csv("../data/processed/hourly.csv", index_col=0, parse_dates=True)
F = fl.build_features(h)
tscv, models = fl.make_tscv(), fl.MODELS

W, H = 168, 24                      # stated in advance: 7-day window in, next 24 h out
y_arr = F["actual"].to_numpy()
idx = F.index
hours = idx.hour.to_numpy(); dow = idx.day_of_week.to_numpy()
cal = np.column_stack([np.sin(2*np.pi*hours/24), np.cos(2*np.pi*hours/24),
                       (dow >= 5).astype(float),
                       np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7)])
N_CAL = cal.shape[1]

def make_samples(starts):
    """starts: positions of the first target hour. Window = the 168 h before it."""
    Xw, Xf, Y, M = [], [], [], []
    for i in starts:
        win = y_arr[i - W:i]
        m = win.mean()
        Xw.append(np.column_stack([win / m, cal[i - W:i]]))
        Xf.append(cal[i:i + H])
        Y.append(y_arr[i:i + H] / m)
        M.append(m)
    return (np.array(Xw, np.float32), np.array(Xf, np.float32),
            np.array(Y, np.float32), np.array(M, np.float32))

In [2]:
class Net(nn.Module):
    def __init__(self, cell="gru", hid=32):
        super().__init__()
        rnn = nn.GRU if cell == "gru" else nn.LSTM
        self.rnn = rnn(1 + N_CAL, hid, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hid + H * N_CAL, 64), nn.ReLU(), nn.Linear(64, H))
    def forward(self, xw, xf):
        _, hn = self.rnn(xw)
        hn = hn[0] if isinstance(hn, tuple) else hn      # LSTM returns (h, c)
        return self.head(torch.cat([hn[-1], xf.flatten(1)], dim=1))

def fit_predict(cell, tr_starts, te_starts, seeds=(0, 1, 2), epochs=60, bs=16):
    Xw, Xf, Y, _ = make_samples(tr_starts)
    Xw, Xf, Y = map(torch.tensor, (Xw, Xf, Y))
    Xw_e, Xf_e, _, M_e = make_samples(te_starts)       # test targets are never used here
    preds = []
    for s in seeds:
        torch.manual_seed(s)
        net = Net(cell)
        opt = torch.optim.Adam(net.parameters(), lr=1e-3)
        loss_fn = nn.L1Loss()
        for _ in range(epochs):
            net.train()
            perm = torch.randperm(len(Xw))
            for b in range(0, len(perm), bs):
                j = perm[b:b + bs]
                opt.zero_grad()
                loss_fn(net(Xw[j], Xf[j]), Y[j]).backward()
                opt.step()
        net.eval()
        with torch.no_grad():
            preds.append(net(torch.tensor(Xw_e), torch.tensor(Xf_e)).numpy())
    return np.mean(preds, axis=0) * M_e[:, None]        # back to load units

In [3]:
results = {}
for cell in ("gru", "lstm"):
    oofs = []
    for k, (tr, te) in enumerate(tscv.split(F), start=1):
        te0 = te[0]
        assert hours[te0] == 0
        tr_starts = [i for i in range(W, te0 - H + 1) if hours[i] == 0]   # target fully before the test fold
        te_starts = list(range(te0, te0 + len(te), 24))
        p = fit_predict(cell, tr_starts, te_starts)
        oofs.append(pd.DataFrame({"fold": k, "pred": p.ravel()}, index=idx[te]))
        if k == 1:
            print(cell, "fold-1 training samples:", len(tr_starts))
    results[cell] = pd.concat(oofs)

def mae_by_fold(oof):
    d = F[["actual"]].join(oof, how="inner")
    return (d["pred"] - d["actual"]).abs().groupby(d["fold"]).mean()

runs = {"baseline": fl.oof_from_col(F, tscv, "daytype_lag"),
        "lgbm_noweather": fl.cv_predict_rel(F, tscv, models["lgbm"], fl.REL_NOWX),
        "gru": results["gru"], "lstm": results["lstm"]}
cmp = pd.DataFrame({k: mae_by_fold(v) for k, v in runs.items()})
ex2 = cmp.drop(index=2).mean()
cmp.loc["mean"] = cmp.mean()
cmp.loc["mean_ex_fold2"] = ex2
print(cmp.round(0))

gru fold-1 training samples: 189


lstm fold-1 training samples: 189


               baseline  lgbm_noweather     gru    lstm
fold                                                   
1                2307.0          2406.0  4285.0  4739.0
2                4082.0          3131.0  4201.0  4261.0
3                1608.0          1416.0  2889.0  3027.0
4                1200.0          1017.0  3680.0  3223.0
5                2175.0          1872.0  3518.0  3661.0
6                1281.0          1093.0  2299.0  2879.0
mean             2109.0          1822.0  3479.0  3632.0
mean_ex_fold2    1714.0          1561.0  3334.0  3506.0


In [4]:
# Diagnostic: GRU on fold 1, same settings as before, training error only
tr, te = list(tscv.split(F))[0]
te0 = te[0]
tr_starts = [i for i in range(W, te0 - H + 1) if hours[i] == 0]
Xw, Xf, Y, M = make_samples(tr_starts)
Xw_t, Xf_t, Y_t = map(torch.tensor, (Xw, Xf, Y))

flat = np.abs((1.0 - Y) * M[:, None]).mean()      # reference: forecast = flat 7-day mean
print("reference, flat 7-day mean (train MAE):", round(flat))

torch.manual_seed(0)
net = Net("gru")
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
loss_fn = nn.L1Loss()
for ep in range(1, 61):
    net.train()
    perm = torch.randperm(len(Xw_t))
    for b in range(0, len(perm), 16):
        j = perm[b:b + 16]
        opt.zero_grad()
        loss_fn(net(Xw_t[j], Xf_t[j]), Y_t[j]).backward()
        opt.step()
    if ep in (1, 5, 10, 20, 40, 60):
        net.eval()
        with torch.no_grad():
            p = net(Xw_t, Xf_t).numpy()
        print(f"epoch {ep:>2}: train MAE = {np.abs((p - Y) * M[:, None]).mean():.0f}")
print("fold-1 test MAE from before (3-seed GRU): 4285")

reference, flat 7-day mean (train MAE): 12514


epoch  1: train MAE = 44755


epoch  5: train MAE = 4670


epoch 10: train MAE = 3925


epoch 20: train MAE = 4187


epoch 40: train MAE = 3738


epoch 60: train MAE = 3835
fold-1 test MAE from before (3-seed GRU): 4285


In [5]:
def make_samples2(starts):
    Xw, Xf, Y, M, B = [], [], [], [], []
    for i in starts:
        win = y_arr[i - W:i]
        m = win.mean()
        Xw.append(np.column_stack([win / m, cal[i - W:i]]))
        Xf.append(cal[i:i + H])
        Y.append(y_arr[i:i + H] / m)
        M.append(m)
        # day-type baseline as ratios: yesterday for Tue-Fri, same day last week otherwise
        B.append((win[144:168] if dow[i] in (1, 2, 3, 4) else win[0:24]) / m)
    return tuple(np.array(a, np.float32) for a in (Xw, Xf, Y, M, B))

class ResNet(nn.Module):
    def __init__(self, cell="gru", hid=32):
        super().__init__()
        rnn = nn.GRU if cell == "gru" else nn.LSTM
        self.rnn = rnn(1 + N_CAL, hid, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hid + H * N_CAL, 64), nn.ReLU(), nn.Linear(64, H))
        nn.init.zeros_(self.head[-1].weight); nn.init.zeros_(self.head[-1].bias)  # starts at the baseline
    def forward(self, xw, xf, base):
        _, hn = self.rnn(xw)
        hn = hn[0] if isinstance(hn, tuple) else hn
        return base + self.head(torch.cat([hn[-1], xf.flatten(1)], dim=1))

# fold 1: fit on training days, validate on the last 28 training days
tr, te = list(tscv.split(F))[0]
te0 = te[0]
all_starts = [i for i in range(W, te0 - H + 1) if hours[i] == 0]
fit_starts, val_starts = all_starts[:-28], all_starts[-28:]

def fit_val(cell, seeds=(0, 1, 2), epochs=60, bs=16, lr=1e-3):
    Xw, Xf, Y, _, B = map(torch.tensor, make_samples2(fit_starts))
    Xw_v, Xf_v, _, M_v, B_v = make_samples2(val_starts)
    preds = []
    for s in seeds:
        torch.manual_seed(s)
        net = ResNet(cell)
        opt = torch.optim.Adam(net.parameters(), lr=lr)
        lf = nn.L1Loss()
        for _ in range(epochs):
            net.train()
            perm = torch.randperm(len(Xw))
            for b in range(0, len(perm), bs):
                j = perm[b:b + bs]
                opt.zero_grad()
                lf(net(Xw[j], Xf[j], B[j]), Y[j]).backward()
                opt.step()
        net.eval()
        with torch.no_grad():
            preds.append(net(torch.tensor(Xw_v), torch.tensor(Xf_v), torch.tensor(B_v)).numpy())
    return np.mean(preds, axis=0) * M_v[:, None]

_, _, Y_v, M_v, B_v = make_samples2(val_starts)
actual_v = Y_v * M_v[:, None]

# LightGBM on the same split: fit on rows before the validation block
v0 = val_starts[0]; vend = v0 + 28 * 24
Xg = F.iloc[:v0][fl.REL_NOWX]
yg = F.iloc[:v0]["actual"] / F.iloc[:v0]["roll_mean_7d"]
ok = Xg.notna().all(axis=1) & yg.notna()
gb = models["lgbm"]().fit(Xg[ok], yg[ok])
p_gb = gb.predict(F.iloc[v0:vend][fl.REL_NOWX]) * F.iloc[v0:vend]["roll_mean_7d"].to_numpy()

res = {"baseline": np.abs(B_v * M_v[:, None] - actual_v).mean(),
       "lgbm": np.abs(p_gb - F.iloc[v0:vend]["actual"].to_numpy()).mean()}
for cell in ("gru", "lstm"):
    res[cell + "_resid"] = np.abs(fit_val(cell) - actual_v).mean()
print(pd.Series(res).round(0))

baseline      3711.0
lgbm          3200.0
gru_resid     3630.0
lstm_resid    3636.0
dtype: float64


## Conclusion
- Plain GRU / LSTM: mean MAE 3,479 / 3,632 vs 2,109 for the baseline and 1,822 for LightGBM, worse than the baseline in all six folds.
- Training-error check: the network underfit (train MAE about 3,800, close to its test MAE), it did not overfit.
- Reparameterised to start from the day-type baseline, on a held-out 28-day validation block: baseline 3,711, LightGBM 3,200, GRU 3,630, LSTM 3,636. A rule set in advance (continue only if within 5% of LightGBM) stopped the run, so the six-fold run was not done.
- Limits: 189 to 336 daily samples, untuned hyperparameters, one architecture, one validation block. This supports "not better under this setup", not "deep learning does not work here".
- Not used in the final model.